<div >
<img src = "../banner.jpg" />
</div>

<a target="_blank" href="https://colab.research.google.com/github/ignaciomsarmiento/BDML_202610/blob/main/Lecture08/Notebook_NB.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



# Naive Bayes: Ejemplo Paso a Paso

**Naive Bayes** es un clasificador probabilístico basado en el teorema de Bayes. 



\begin{align}
Pr (Y=1|X)=\frac{\overbrace{f(X|Y=1)}^{\text{verosimilitud}} \cdot \overbrace{\pi(Y=1)}^{\text{prior}}}{\underbrace{f(X|Y=1)\pi(Y=1) + f(X|Y=0)(1-\pi(Y=1))}_{\text{probabilidad total (normalización)}}}
\end{align}

El supuesto **"naive"** (ingenuo) es que los predictores son **condicionalmente independientes** dado $Y=k$:

$$f(X|Y=k) = f(X_1|Y=k) \times f(X_2|Y=k) \times \cdots \times f(X_p|Y=k)$$

Esto simplifica enormemente la estimación: en lugar de modelar la distribución conjunta de todos los predictores, estimamos $p$ distribuciones univariadas, una por predictor y por clase.



## Ejemplo (ISLR Fig. 4.10)

Tenemos $p = 3$ predictores y $K = 2$ clases:

- $X_1, X_2$: predictores cuantitativos (continuos) → modelados con distribuciones normales.
- $X_3$: predictor cualitativo con 3 niveles → modelado con proporciones por clase.
- Prior: $\pi(Y=1) = \pi(Y=2) = 0.5$ (clases igualmente frecuentes).

La notación $\hat{f}_{kj}$ denota la **densidad estimada del predictor $j$ dentro de la clase $k$**. Por ejemplo, $\hat{f}_{12}$ es la densidad de $X_2$ entre las observaciones de clase 1.

<div >
<img width="500" height="300" src = "figs/toy_example.png" />
</div>

Queremos clasificar una nueva observación 


$$x^* = (0.4,\ 1.5,\ 1)$$



**Paso 1:** Evaluar $\hat{f}_{kj}(x^*_j)$ para cada predictor $j$ y cada clase $k$, es decir, ¿qué tan "probable" es el valor observado de cada predictor si la observación pertenece a la clase $k$?

### Clase 1

Del gráfico "adivinamos" que $X_1|Y=1 \sim N(0,1)$. La densidad normal es:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\,e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

Evaluamos en $x_1^* = 0.4$:

In [1]:
# Implementación manual de la densidad normal — para ver de dónde viene el número
normal_density <- function(x, mu = 0, sd = 1) {
  1 / (sqrt(2 * pi) * sd) * exp(-0.5 * ((x - mu) / sd)^2)
}

normal_density(0.4)  # f_11(0.4): X1 = 0.4 bajo clase 1 ~ N(0,1)

[1] 0.3682701

In [2]:
# R ya tiene dnorm() incorporado — resultado idéntico
dnorm(0.4, mean = 0, sd = 1)  # f_11(0.4) = 0.368 ✓

[1] 0.3682701

Del gráfico, los valores para **clase 1** son:

| Predictor | Distribución | $\hat{f}_{1j}(x^*_j)$ |
|---|---|---|
| $X_1 = 0.4$ | $N(0, 1)$ | $\hat{f}_{11}(0.4) = 0.368$ |
| $X_2 = 1.5$ | $N(2, 1.5)$ | $\hat{f}_{12}(1.5) = 0.484$ |
| $X_3 = 1$   | Categórica  | $\hat{f}_{13}(1) = 0.226$  |

Y para **clase 2**:

| Predictor | $\hat{f}_{2j}(x^*_j)$ |
|---|---|
| $X_1 = 0.4$ | $\hat{f}_{21}(0.4) = 0.030$ |
| $X_2 = 1.5$ | $\hat{f}_{22}(1.5) = 0.130$ |
| $X_3 = 1$   | $\hat{f}_{23}(1) = 0.616$  |

Nota: los valores de $X_3$ son proporciones empíricas (frecuencia del nivel 1 dentro de cada clase), no densidades continuas. Naive Bayes los trata de la misma forma.

**Paso 2:** Aplicar el supuesto de independencia condicional para obtener la verosimilitud de cada clase.

$$\hat{f}(X|Y=1) = \hat{f}_{11}(0.4)\times \hat{f}_{12}(1.5)\times \hat{f}_{13}(1)$$

$$\hat{f}(X|Y=2) = \hat{f}_{21}(0.4)\times \hat{f}_{22}(1.5)\times \hat{f}_{23}(1)$$

**Paso 3:** Aplicar el Teorema de Bayes para obtener la probabilidad posterior de cada clase.

In [3]:
# Verosimilitud clase 1: producto de las tres densidades individuales
f11 <- 0.368
f12 <- 0.484
f13 <- 0.226

f1 <- f11 * f12 * f13
f1

[1] 0.04025331

In [4]:
# Verosimilitud clase 2
f21 <- 0.030
f22 <- 0.130
f23 <- 0.616

f2 <- f21 * f22 * f23
f2

[1] 0.0024024

In [5]:
p1 <- 0.5  # prior: clases equiprobables

In [6]:
# Pr(Y=1 | x*) — probabilidad posterior de clase 1
prob_clase1 <- f1 * p1 / (f1 * p1 + f2 * (1 - p1))
prob_clase1

[1] 0.9436793

In [7]:
# Pr(Y=2 | x*) — debe sumar 1 con prob_clase1
prob_clase2 <- f2 * (1 - p1) / (f1 * p1 + f2 * (1 - p1))
prob_clase2

[1] 0.05632071

## Conclusión

$Pr(Y=1|x^*) \approx 0.944$ vs $Pr(Y=2|x^*) \approx 0.056$ → el clasificador asigna $x^*$ a la **clase 1**.

¿Por qué ganó la clase 1 a pesar de tener priors iguales? Porque su verosimilitud ($f_1 \approx 0.040$) es mucho mayor que la de la clase 2 ($f_2 \approx 0.0024$): los valores $x_1^*=0.4$ y $x_2^*=1.5$ son mucho más típicos de la clase 1 que de la clase 2.

**Nota sobre el supuesto "naive":** En este ejemplo, los tres predictores son independientes por construcción. En la práctica, esto rara vez se cumple, pero Naive Bayes funciona sorprendentemente bien incluso cuando se viola, especialmente cuando el número de predictores es grande y los datos son escasos.